# **ClinTrialPredict: ETL Gatekeeper & Master Forensic Audit (v4.0)**

**Data Source**: [AACT Downloads](https://aact.ctti-clinicaltrials.org/downloads) (Flat Text Files button -> Download File).

### **Updates in v4.0 (The Complete Ingestion Anchor):**
- **Full 19-File Audit**: tracks all critical, optional, and EDA files from the raw AACT export.
- **Lifecycle Forensic**: Segments vacancy between Historical (Frozen) and Ongoing (Active) trials.
- **360° Column Tracking**: Monitors 38 functional fields across 6 logical groups (Identity, LLM Fuel, Filters, Model, Timeline, Secondary).

## **1. Master Field Registry & functional Roles**

| Group | Fields | Mechanical Role |
| :--- | :--- | :--- |
| **A: Alpha Identity** | `official_title`, `brief_title`, `interventions_name`, `other_names` | Resolves drug codes to brand/generic names. |
| **B: Target/Molecular**| `intervention_desc`, `summary`, `conditions`, `mesh_terms` | Input for LLM to determine Mechanism of Action (MOA). |
| **C: Clinical Filters** | `status`, `phase`, `modality`, `sponsor_class`, `expanded_access` | Defines the Industry-led Phase 2/3 drug universe. |
| **D: Model Pillars** | `allocation`, `masking`, `model`, `purpose`, `arms`, `dmc`, `facilities` | Direct predictive features for the XGBoost model. |
| **E: Timeline/Geo** | `start_date`, `primary_completion`, `countries` | Temporal windowing and geographic footprint. |
| **F: Secondary** | `p_value`, `criteria`, `age`, `enrollment`, `descriptions` | Demographic context and forensic validation. |

---

## **2. Environment Setup**
We initialize paths relative to the project root and define the reporting buffer.

In [1]:
import pandas as pd
import numpy as np
import csv
import sys
import os
from datetime import datetime
from pathlib import Path
import warnings
warnings.filterwarnings('always', category=pd.errors.ParserWarning)
pd.set_option('future.no_silent_downcasting', True)

current_dir = Path.cwd()
project_root = current_dir
while not (project_root / 'src').exists(): project_root = project_root.parent

RAW_DATA_PATH = project_root / "data" / "raw"
OUTPUT_PATH = project_root / "data"
LOG_DIR = project_root / "data" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
REPORT_FILE = LOG_DIR / f"etl_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

def get_raw_line_count(filepath):
    """Reads physical line count on disk to detect ingestion loss."""
    try:
        with open(filepath, 'rb') as f: return sum(1 for _ in f) - 1
    except: return 0

## **3. Master Manifest & Pillar Registry**
This section defines the 19 required files and the functional grouping of the 38 fields we audit.

In [2]:
MANIFEST = [
    {"file": "studies.txt", "critical": True, "unique_id": "nct_id"},
    {"file": "sponsors.txt", "critical": True, "unique_id": None},
    {"file": "interventions.txt", "critical": True, "unique_id": None},
    {"file": "designs.txt", "critical": True, "unique_id": "nct_id"},
    {"file": "eligibilities.txt", "critical": True, "unique_id": "nct_id"},
    {"file": "brief_summaries.txt", "critical": True, "unique_id": "nct_id"},
    {"file": "design_outcomes.txt", "critical": True, "unique_id": None},
    {"file": "outcome_analyses.txt", "critical": True, "unique_id": None},
    {"file": "calculated_values.txt", "critical": True, "unique_id": "nct_id"},
    {"file": "conditions.txt", "critical": True, "unique_id": None},
    {"file": "intervention_other_names.txt", "critical": True, "unique_id": None},
    {"file": "countries.txt", "critical": True, "unique_id": None},
    {"file": "design_groups.txt", "critical": True, "unique_id": None},
    {"file": "browse_interventions.txt", "critical": True, "unique_id": None},
    {"file": "browse_conditions.txt", "critical": True, "unique_id": None},
    {"file": "detailed_descriptions.txt", "critical": False, "unique_id": "nct_id"},
    {"file": "baseline_measurements.txt", "critical": False, "unique_id": None},
    {"file": "outcome_measurements.txt", "critical": False, "unique_id": None},
    {"file": "provided_documents.txt", "critical": False, "unique_id": None}
]

GROUPED_PILLARS = {
    "Group_A_AlphaIdentity": {"studies.txt": ['official_title', 'brief_title', 'acronym'], "interventions.txt": ['name'], "intervention_other_names.txt": ['name']},
    "Group_B_TargetMolecular": {"interventions.txt": ['description'], "brief_summaries.txt": ['description'], "conditions.txt": ['name'], "browse_interventions.txt": ['mesh_term'], "browse_conditions.txt": ['mesh_term']},
    "Group_C_ClinicalFilters": {"studies.txt": ['overall_status', 'phase', 'study_type', 'why_stopped', 'has_expanded_access'], "sponsors.txt": ['agency_class', 'lead_or_collaborator'], "interventions.txt": ['intervention_type']},
    "Group_D_ModelPillars": {"designs.txt": ['allocation', 'masking', 'intervention_model', 'primary_purpose'], "studies.txt": ['number_of_arms', 'has_dmc'], "eligibilities.txt": ['healthy_volunteers', 'gender'], "design_outcomes.txt": ['measure'], "calculated_values.txt": ['number_of_facilities']},
    "Group_E_TimelineGeo": {"studies.txt": ['start_date', 'primary_completion_date'], "countries.txt": ['name']},
    "Group_F_Secondary": {"outcome_analyses.txt": ['p_value'], "eligibilities.txt": ['criteria', 'minimum_age', 'maximum_age'], "studies.txt": ['enrollment'], "detailed_descriptions.txt": ['description']}
}

SPECIAL_CONFIG = {
    "studies.txt": {"date_cols": ["study_first_submitted_date", "results_first_submitted_date", "completion_date", "start_date", "primary_completion_date"]},
    "outcome_analyses.txt": {"numeric_cols": ["p_value"]}
}

## **4. Master Forensic Auditor**
This tracker maintains state trial-by-trial to detect horizontal missingness and segments the report by trial lifecycle (Historical vs. Ongoing).

In [3]:
class MasterForensicAuditor:
    def __init__(self, grouped_pillars):
        self.registry = pd.DataFrame()
        self.grouped_pillars = grouped_pillars
        self.lifecycle_meta = pd.DataFrame()

    def update(self, df, filename):
        relevant_cols = []
        for group, files in self.grouped_pillars.items():
            if filename in files: relevant_cols.extend(files[filename])
        if not relevant_cols or 'nct_id' not in df.columns: return

        if filename == 'studies.txt':
            self.lifecycle_meta = df.groupby('nct_id')[['overall_status', 'start_date']].first()

        present = df.groupby('nct_id')[list(set(relevant_cols))].first().notna()
        present.columns = [f"{filename.split('.')[0]}_{c}" for c in present.columns]
        if self.registry.empty: self.registry = present
        else: self.registry = self.registry.combine_first(present)

    def get_lifecycle_report(self, target_ids=None):
        reg = self.registry if target_ids is None else self.registry[self.registry.index.isin(target_ids)]
        meta = self.lifecycle_meta[self.lifecycle_meta.index.isin(reg.index)]
        if reg.empty: return "No data.", None

        final_statuses = ['COMPLETED', 'TERMINATED', 'WITHDRAWN']
        hist_mask = meta['overall_status'].str.upper().isin(final_statuses)
        reg_hist = reg[hist_mask]; reg_ongoing = reg[~hist_mask]

        def get_vacancy(sub_reg):
            if sub_reg.empty: return pd.Series()
            return (sub_reg.fillna(False).astype(bool) == False).mean() * 100

        combined_vac = pd.DataFrame({"Vacancy_Historical%": get_vacancy(reg_hist), "Vacancy_Ongoing%": get_vacancy(reg_ongoing)})

        # Readiness Logic (Mandatory Paths)
        mandatory_llm = ['studies_brief_title', 'brief_summaries_description', 'interventions_name']
        mandatory_xgb = ['studies_overall_status', 'studies_phase', 'sponsors_agency_class', 'interventions_intervention_type']

        def count_ready(sub_reg, cols):
            if sub_reg.empty: return 0
            sub_bool = sub_reg.fillna(False).astype(bool)
            return (sub_bool[sub_bool.columns.intersection(cols)] == True).all(axis=1).sum()

        report = f"MASTER LIFECYCLE SUMMARY:\n"
        report += f"- HISTORICAL (Closed): {len(reg_hist):,} trials. LLM-Ready: {count_ready(reg_hist, mandatory_llm):,}, XGB-Ready: {count_ready(reg_hist, mandatory_xgb):,}\n"
        report += f"- ONGOING (Active):    {len(reg_ongoing):,} trials. LLM-Ready: {count_ready(reg_ongoing, mandatory_llm):,}, XGB-Ready: {count_ready(reg_ongoing, mandatory_xgb):,}\n"
        return report, combined_vac

auditor = MasterForensicAuditor(GROUPED_PILLARS)

## **5. Surgical Cleaning Utilities**
Standard normalization for century typos and numeric noise.

In [4]:
report_buffer = []
def log(msg): print(msg); report_buffer.append(msg)
def clean_numeric(series):
    clean = series.astype(str).str.replace(r'\s+', '', regex=True).str.replace(r'[,<>]', '', regex=True).str.replace('%', '', regex=False)
    nums = pd.to_numeric(clean, errors='coerce')
    if series.name == 'p_value': nums = nums.clip(0.0, 1.0)
    return nums
def clean_date(series):
    series = series.astype(str).str.replace(r'^10(\d{2}-\d{2}-\d{2})', r'20\1', regex=True)
    dates = pd.to_datetime(series, errors='coerce')
    return dates.where((dates.dt.year >= 1900) & (dates.dt.year <= 2100), pd.NaT)

## **6. Iron Gate Batch Execution**
Processes every file in the manifest, captures structural loss, and updates the horizontal tracker.

In [5]:
LOAD_PARAMS = {"sep": "|", "dtype": str, "quotechar": '"', "quoting": csv.QUOTE_MINIMAL, "low_memory": False, "on_bad_lines": "warn"}
results = []
for entry in MANIFEST:
    f = entry['file']; in_p = RAW_DATA_PATH / f; out_p = OUTPUT_PATH / f
    if not in_p.exists(): continue
    raw_lines = get_raw_line_count(in_p)
    log(f"\nIngesting: {f}...")
    try:
        df = pd.read_csv(in_p, **LOAD_PARAMS)
        for col in df.select_dtypes(include=['object']): df[col] = df[col].str.strip()
        if f in SPECIAL_CONFIG:
            cfg = SPECIAL_CONFIG[f]
            for col in cfg.get("numeric_cols", []): df[col] = clean_numeric(df[col])
            for col in cfg.get("date_cols", []): df[col] = clean_date(df[col])
        auditor.update(df, f)
        if entry['unique_id'] and entry['unique_id'] in df.columns: df.drop_duplicates(entry['unique_id'], inplace=True)
        df.to_csv(out_p, index=False, sep='|', quoting=csv.QUOTE_MINIMAL)
        results.append({"File": f, "DiskLines": raw_lines, "CleanRows": len(df), "StructLoss": raw_lines - len(df)})
    except Exception as e: log(f"   [ERROR] {e}")
df_loss = pd.DataFrame(results)
display(df_loss)


Ingesting: studies.txt...

Ingesting: sponsors.txt...

Ingesting: interventions.txt...

Ingesting: designs.txt...

Ingesting: eligibilities.txt...

Ingesting: brief_summaries.txt...

Ingesting: design_outcomes.txt...

Ingesting: outcome_analyses.txt...

Ingesting: calculated_values.txt...

Ingesting: conditions.txt...

Ingesting: intervention_other_names.txt...

Ingesting: countries.txt...

Ingesting: design_groups.txt...

Ingesting: browse_interventions.txt...

Ingesting: browse_conditions.txt...

Ingesting: detailed_descriptions.txt...

Ingesting: baseline_measurements.txt...

Ingesting: outcome_measurements.txt...


,File,DiskLines,CleanRows,StructLoss
0,studies.txt,582278,582278,0
1,sponsors.txt,930094,930094,0
2,interventions.txt,984317,984317,0
3,designs.txt,577519,577519,0
4,eligibilities.txt,581307,581307,0
5,brief_summaries.txt,581307,581307,0
6,design_outcomes.txt,3610398,3610398,0
7,outcome_analyses.txt,318181,318181,0
8,calculated_values.txt,582278,582278,0
9,conditions.txt,1038946,1038946,0


## **7. Final Master Forensic Audit & Lifecycle Report**
Applying all 7 production filters and auditing horizontal vacancy by lifecycle segment.

In [6]:
log("\nFiltering for Production Target Cohort...")
df_studies = pd.read_csv(OUTPUT_PATH / 'studies.txt', sep='|')
leads = pd.read_csv(OUTPUT_PATH / 'sponsors.txt', sep='|', usecols=['nct_id', 'lead_or_collaborator', 'agency_class'])
leads = leads[(leads['lead_or_collaborator'].str.upper() == 'LEAD') & (leads['agency_class'].str.upper() == 'INDUSTRY')]['nct_id'].unique()
drug_ids = pd.read_csv(OUTPUT_PATH / 'interventions.txt', sep='|', usecols=['nct_id', 'intervention_type'])
drug_ids = drug_ids[drug_ids['intervention_type'].str.upper().isin(['DRUG', 'BIOLOGICAL', 'GENETIC'])]['nct_id'].unique()

df_f = df_studies[df_studies['nct_id'].isin(leads) & df_studies['nct_id'].isin(drug_ids)].copy()
df_f = df_f[df_f['study_type'].str.upper() == 'INTERVENTIONAL']
df_f = df_f[~df_f['phase'].fillna('NA').str.upper().isin(['EARLY_PHASE1', 'PHASE1', 'PHASE4', 'NA'])].dropna(subset=['phase'])
allowed = ['COMPLETED', 'TERMINATED', 'WITHDRAWN', 'RECRUITING', 'ACTIVE, NOT RECRUITING', 'NOT YET RECRUITING', 'ENROLLING BY INVITATION']
df_f = df_f[df_f['overall_status'].str.upper().isin(allowed)]
if 'has_expanded_access' in df_f.columns: df_f = df_f[df_f['has_expanded_access'].fillna('f').str.lower().isin(['f', 'false', '0', 'no'])]
mask_covid = df_f['why_stopped'].fillna('').astype(str).str.lower().apply(lambda x: any(k in x for k in ['covid', 'pandemic']))
df_f = df_f[~mask_covid]

target_ids = set(df_f['nct_id'])
summary, lifecycle_vac = auditor.get_lifecycle_report(target_ids=target_ids)

log("\n" + "="*60 + "\nMASTER FORENSIC LIFECYCLE AUDIT (v4.0)\n" + "="*60)
log(summary)
log("\nCOMPARATIVE VACANCY BY SEGMENT:")
for group, files in GROUPED_PILLARS.items():
    log(f"\n--- {group} ---")
    grp_cols = [f"{f.split('.')[0]}_{c}" for f, cols in files.items() for c in cols]
    log(lifecycle_vac[lifecycle_vac.index.isin(grp_cols)].to_string())

with open(REPORT_FILE, 'w') as f_out: f_out.write("\n".join(report_buffer))
log(f"\n[SUCCESS] Master ETL Complete. Log: {REPORT_FILE}")


Filtering for Production Target Cohort...


/tmp/ipykernel_4557/4048111601.py:2: DtypeWarning: Columns (46,47,48,53,68) have mixed types. Specify dtype option on import or set low_memory=False.
  df_studies = pd.read_csv(OUTPUT_PATH / 'studies.txt', sep='|')



MASTER FORENSIC LIFECYCLE AUDIT (v4.0)
MASTER LIFECYCLE SUMMARY:
- HISTORICAL (Closed): 39,408 trials. LLM-Ready: 39,408, XGB-Ready: 39,408
- ONGOING (Active):    4,134 trials. LLM-Ready: 4,134, XGB-Ready: 4,134


COMPARATIVE VACANCY BY SEGMENT:

--- Group_A_AlphaIdentity ---
                               Vacancy_Historical%  Vacancy_Ongoing%
intervention_other_names_name            56.293138         62.022254
interventions_name                        0.000000          0.000000
studies_acronym                          80.511571         70.803096
studies_brief_title                       0.000000          0.000000
studies_official_title                    1.768676          0.000000

--- Group_B_TargetMolecular ---
                                Vacancy_Historical%  Vacancy_Ongoing%
brief_summaries_description                0.000000          0.000000
browse_conditions_mesh_term               11.845311         21.432027
browse_interventions_mesh_term            23.898701         48.06